In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdulsalamramatu/100-feature-response-sample/total_response_sample.xlsx
/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv


In [2]:
!pip install replicate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.7 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import pandas as pd
from tqdm import tqdm
import time
from kaggle_secrets import UserSecretsClient
import replicate
from typing import Dict, List


# 100 response per feature extraction

In [4]:
math_df=pd.read_csv("/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv")
mtld_na=math_df.loc[math_df.mtld.isna()].conversation_id.tolist()
math_df=math_df[~math_df['conversation_id'].isin(mtld_na)]

math_df["mtld_log"] = np.log1p(math_df["mtld"])
math_df["word_count_log"] = np.log1p(math_df["word_count"])

In [5]:
def sample_score_bins(df, score, n_total=100, n_bins=4):
    bin_edges=np.percentile(df[score], np.linspace(0,100, n_bins+1))
    bin_edges[0]=-np.inf
    bin_edges[-1]=np.inf
    df['score_bin']=pd.cut(
        df[score], 
        bins=bin_edges, 
        labels=[f'Bin{i+1}' for i  in range(n_bins)],
        include_lowest=True)
    
    samples_per_bin=n_total// n_bins
    reminder=n_total % n_bins
    
    sampled_dfs=[]
    
    for i, bin_label in enumerate(df['score_bin'].cat.categories):
        
        bin_df=df[df['score_bin']==bin_label]
        n_samples=samples_per_bin +(1 if i < reminder else 0)
        n_samples=min(n_samples, len(bin_df))
        
        if n_samples>0:
            sampled_dfs.append(bin_df.sample(n=n_samples, random_state=42))

    sampled_df = pd.concat(sampled_dfs)
    cols_to_keep=['tutor', 'conversation_id', 'student_mistake',  'tutors_response']

    columns_to_keep = list(set(cols_to_keep ))
    sampled_df = sampled_df[columns_to_keep]
    sampled_df['conversation_id'] = sampled_df['conversation_id'].astype(str)

    #sampled_df = sampled_df.drop(columns=['score_bin'])
    
    return sampled_df, bin_edges

In [6]:
features = ['PressReasoning_prob', 'PressAccuracy_prob', 'Uptake_prob', 'politeness_score', 'agency_score']
sample_dict={}
bin_edges_dict={}
for feature in features:
    print(f"Sampling for {feature}....")
    sampled_df, edges= sample_score_bins(math_df, feature,n_total=100, n_bins=4)
    sample_dict[feature]=sampled_df
    bin_edges_dict[feature]=edges   
    sampled_df.to_csv(f"sampled_{feature}.csv", index=False)

Sampling for PressReasoning_prob....
Sampling for PressAccuracy_prob....
Sampling for Uptake_prob....
Sampling for politeness_score....
Sampling for agency_score....


In [7]:
math_df

,tutor,conversation_id,student_mistake,tutors_response,PressReasoning_prob,PressAccuracy_prob,Uptake_prob,politeness_score,agency_score,flesch_reading_ease,...,word_count,flesch_score,flesch_ease,Mistake_Identification,Mistake_Location,Actionability,Providing_Guidance,mtld_log,word_count_log,score_bin
0,Sonnet,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"Great, you've correctly identified the cost of...",0.000816,0.009659,0.558105,0.975984,0.577917,52.050000,...,26,52.0500,16.0,2,2,2,2,3.393613,3.295837,Bin4
1,Llama318B,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,Now that we know the cost of 1 pound of meat i...,0.000762,0.005924,0.908691,0.951331,0.540315,80.097647,...,32,80.0976,6.0,2,1,1,1,3.606584,3.496508,Bin3
2,Llama31405B,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"You're close, but I notice that you calculated...",0.004723,0.757324,0.013527,0.907285,0.487226,47.376429,...,39,47.3764,18.0,2,2,2,2,3.483349,3.688879,Bin1
3,GPT4,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"That's correct. So, if 1 pound of meat costs $...",0.001601,0.690430,0.047394,0.986556,0.510410,98.252500,...,27,98.2525,2.0,2,2,2,2,3.450305,3.332205,Bin2
4,Mistral,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,It seems like you've calculated the cost as if...,0.000623,0.004658,0.633301,0.302074,0.499444,72.665000,...,28,72.6650,13.0,2,2,2,2,4.707366,3.367296,Bin2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2471,Mistral,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,It seems there might be a misunderstanding in ...,0.000394,0.010368,0.562988,0.963009,0.461775,32.434286,...,22,32.4343,14.0,2,2,2,1,3.135494,3.135494,Bin1
2472,Phi3,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,"To solve this problem, we need to add the numb...",0.000448,0.009674,0.449219,0.657242,0.510343,69.788000,...,21,69.7880,10.0,0,0,0,0,4.138999,3.091042,Bin2
2473,Sonnet,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,That's a great start and I like how you worked...,0.000611,0.007957,0.683105,0.991297,0.586221,60.765000,...,33,60.7650,9.0,2,2,2,2,4.346788,3.526361,Bin4
2474,Expert,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,Okay. So Hector gave 5 less than four times as...,0.000407,0.006321,0.685059,0.129152,0.482296,71.767857,...,26,71.7679,7.0,2,2,2,2,3.659863,3.295837,Bin1


# LLM (Claude) Annotation

In [8]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("tama_replicate_key")

if secret_value_0:
    print('key gotten')
else:
    print('key not gotten')

key gotten


In [12]:
LOG_FILE = "classification_errors.log"
class TutorResponseAnnotator:
    def __init__(self, replicate_api_key: str, model: str= 'claude-3-sonnet-20240229'):
        self.replicate_api_token=replicate_api_key
        self.model=model
        self.client=replicate.Client(api_token=replicate_api_key)
        print(f"Replicate client initialized: {self.client is not None}")
    def create_prompt(self,  tutor_response: str, rubric:str, rubric_def:str, student_mistake:str)-> str:
        options=[0,1]
        prompt= f"""
        CONTEXT: 
        You are an expert annotator for math tutoring dialogue quality.        
        TASK:
        Rate the response on how much **{rubric}** is present in the tutor's response to a student's mistake {student_mistake} , your judgement should be solely based on the definition  below:
        Definition of {rubric}: {rubric_def}
        
        Tutor's response to evaluate:
        {tutor_response}
        
        Ratings Scale:
        1= Response satisfies the definition of {rubric}
        0= Response does not satisfy the definition of {rubric} 


        RESPONSE and  INSTRUCTIONS:
        1. Respond with only one integer rating  0 or 1
        2. No explanations, No additional text """
        


        return prompt.strip()
    def evaluate_response(self,conv_id: str, rubric:str,rubric_def:str, tutor_response: str,student_mistake:str,  max_tries:int =3) -> int:
 

        prompt = self.create_prompt(tutor_response,rubric,rubric_def,student_mistake)
        model_id="anthropic/claude-4.5-sonnet"
        for attempt in range(max_tries):
            try:
                print(f"  Analyzing (attempt {attempt + 1}/{max_tries})...")

                output = self.client.run(model_id,
                            input={
                                  "prompt":prompt,
                                  "max_tokens":1024,
                                  "temperature":0.0,
                                  "system": "You are an expert annotator for math tutoring dialogue quality.  Respond with only 0 or 1."
   
                              })
                response_text = ""
                for item in output:
                    response_text += item
                
                response = response_text.strip()
                
                if response in ['0', '1']:
                    return int(response)
                elif '0' in response:
                    return 0
                elif '1' in response:
                    return 1
                else:
                    print(f"     Unexpected response: '{response}'. Defaulting to 0")
                    return 0
            except Exception as e:
                print(f"     API error: {e}. Retrying...")
                time.sleep(2)
                
        with open(LOG_FILE, "a") as log:
            log.write(f"Retry limit exceeded | conv_id: {conv_id}, rubric: {rubric}, response: '{tutor_response[:100]}...'\n")

        return 0


In [13]:
def process_responses(path,rubrics,api_key):
    definitions={
    'politeness': " Language that conveys respect and consideration towards others",  
    'agency': "Language that conveys initiative, purposeful action, or the capacity to act",
    'PressAccuracy':"Language that seeks to verify, clarify, or improve the accuracy of a statement or claim.",
    'Uptake':" Language that reflects or builds on another person's contribution to support shared understanding and discussion.",
    "PressReasoning":"Language that seeks to understand, explain, or justify the reasoning behind a statement or claim."}
    annotator=TutorResponseAnnotator(api_key)
    results={}
    excel_file=pd.ExcelFile(path)
    for rubric in rubrics:
        print(f"Processing rubric: {rubric}")
        if rubric not in excel_file.sheet_names:
            print(f"Warning: Sheet '{rubric}' not found in {path}")
            print(f"Available sheets: {excel_file.sheet_names}")

            continue
        sample_df=pd.read_excel(path, sheet_name=rubric)
        #sample_df = sample_df.head(2).copy()

        print(f"Loaded {len(sample_df)} responses from sheet '{rubric}'")
            
        #test_sample=sample_df.head(5).copy()
        annotation_column = rubric
        rubric_def=definitions[rubric]
        sample_df[annotation_column] = None
        for idx, row in sample_df.iterrows():
            conv_id=str(row.get('conversation_id', f'row{idx}'))
            student_mistake=row['student_mistake']
            tutor_response=row['tutors_response']
            annotation=annotator.evaluate_response(
                    conv_id=conv_id,
                    rubric=rubric, 
                    rubric_def=rubric_def,
                    tutor_response=tutor_response,  
                    student_mistake=student_mistake,
                    max_tries=3)
            
            
            sample_df.at[idx, annotation_column] = annotation
                #print(f"  {rubric} annotation: {annotation}")
            time.sleep(0.5)
        results[rubric] = sample_df
        output_file = f"annotated_{rubric}.csv"
        sample_df.to_csv(output_file, index=False)
    return results

In [14]:
if __name__ == "__main__":
    user_secrets = UserSecretsClient()
    replicate_api_key = user_secrets.get_secret("tama_replicate_key")
    rubrics = ['politeness', 'agency', 'PressReasoning', 'PressAccuracy', 'Uptake']
    path = "/kaggle/input/datasets/abdulsalamramatu/100-feature-response-sample/total_response_sample.xlsx"
    annotated_results = process_responses(path, rubrics, replicate_api_key)

Replicate client initialized: True
Processing rubric: politeness
Loaded 2 responses from sheet 'politeness'
  Analyzing (attempt 1/3)...
  Analyzing (attempt 1/3)...
Processing rubric: agency
Loaded 2 responses from sheet 'agency'
  Analyzing (attempt 1/3)...
  Analyzing (attempt 1/3)...
Processing rubric: PressReasoning
Loaded 2 responses from sheet 'PressReasoning'
  Analyzing (attempt 1/3)...
  Analyzing (attempt 1/3)...
Processing rubric: PressAccuracy
Loaded 2 responses from sheet 'PressAccuracy'
  Analyzing (attempt 1/3)...
  Analyzing (attempt 1/3)...
Processing rubric: Uptake
Loaded 2 responses from sheet 'Uptake'
  Analyzing (attempt 1/3)...
  Analyzing (attempt 1/3)...
